In [43]:
import pandas as pd

In [44]:
# Loading the Dataset
df=pd.read_csv('dataset/processed/clean_data.csv')

In [45]:
print(df.shape)
df.head()

(180516, 45)


,Type,Days for shipping (real),Days for shipment (scheduled),Benefit per order,Sales per customer,Late_delivery_risk,Category Id,Category Name,Customer City,Customer Country,...,Order Status,Product Card Id,Product Category Id,Product Name,Product Price,Product Status,shipping date (DateOrders),Shipping Mode,delay_gap,order_month
0,CASH,2,4,88.790001,239.979996,0,43,Camping & Hiking,Hickory,EE. UU.,...,CLOSED,957,43,Diamondback Women's Serene Classic Comfort Bi,299.980011,0,2015-01-03 00:00:00,Standard Class,-2,1
1,PAYMENT,3,4,36.470001,107.889999,0,18,Men's Footwear,Chicago,EE. UU.,...,PENDING_PAYMENT,403,18,Nike Men's CJ Elite 2 TD Football Cleat,129.990005,0,2015-01-04 00:21:00,Standard Class,-1,1
2,PAYMENT,3,4,91.180000,193.990005,0,48,Water Sports,Chicago,EE. UU.,...,PENDING_PAYMENT,1073,48,Pelican Sunstream 100 Kayak,199.990005,0,2015-01-04 00:21:00,Standard Class,-1,1
3,PAYMENT,3,4,68.250000,227.500000,0,24,Women's Apparel,Chicago,EE. UU.,...,PENDING_PAYMENT,502,24,Nike Men's Dri-FIT Victory Golf Polo,50.000000,0,2015-01-04 00:21:00,Standard Class,-1,1
4,CASH,5,4,4.100000,40.980000,1,40,Accessories,San Antonio,EE. UU.,...,CLOSED,897,40,Team Golf New England Patriots Putter Grip,24.990000,0,2015-01-06 01:03:00,Standard Class,1,1


In [46]:
# Converting the date columns to datetime
df['order date (DateOrders)']=pd.to_datetime(df['order date (DateOrders)'])
df['shipping date (DateOrders)']=pd.to_datetime(df['shipping date (DateOrders)'])

In [47]:
df['Order_processing_days']=(df['shipping date (DateOrders)']-df['order date (DateOrders)']).dt.days
print(df.Order_processing_days.describe())

count    180516.000000
mean          3.471858
std           1.670481
min           0.000000
25%           2.000000
50%           3.000000
75%           5.000000
max           6.000000
Name: Order_processing_days, dtype: float64


## Feature: order_processing_days

The gap between when an order was placed and when it actually shipped.
A longer processing time likely signals warehouse or supplier bottlenecks
and should be a strong predictor of late delivery.

In [48]:
df['is_holiday_window']=df['order date (DateOrders)'].dt.month.isin([11,12]).astype(int)
print(df['is_holiday_window'].value_counts())

is_holiday_window
0    155254
1     25262
Name: count, dtype: int64


## Feature: is_holiday_window

Binary flag = 1 if the order was placed in November or December.
Even though our EDA showed Q4 is not the peak delay period in this dataset,
we include this feature and let the model decide its importance.

In [49]:
# 0-Monday 6-Sunday
df['Order_day_of_week']=df['order date (DateOrders)'].dt.dayofweek
df['is_weekend_order']=(df['Order_day_of_week']>=5).astype(int)
print(df['Order_day_of_week'].value_counts().sort_index())

Order_day_of_week
0    25786
1    25622
2    25587
3    25750
4    25925
5    25901
6    25945
Name: count, dtype: int64


## Feature: order_day_of_week + is_weekend_order

Orders placed on weekends may face processing delays if warehouses
operate on reduced staff. is_weekend_order is a binary version of this.

In [50]:
# We found this in EDA — encoding manually based on actual late rates
shipping_risk_map={
    'First Class':3,
    "Second Class":2,
    "Standard Class":0,
    "Same Day":1
}
df['Shipping_mode_risk']=df['Shipping Mode'].map(shipping_risk_map)
print(df['Shipping_mode_risk'].value_counts())

Shipping_mode_risk
0    107750
2     35216
3     27813
1      9737
Name: count, dtype: int64


## Feature: shipping_mode_risk

Ordinal encoding based on actual late delivery rates found in EDA.
First Class = 3 (highest risk, 95% late rate)
Second Class = 2 (76% late rate)
Same Day = 1 (45% late rate)
Standard Class = 0 (38% late rate)(lowest risk)

In [51]:
# Aggregate delay history per customer
customer_stats=df.groupby("Order Customer Id").agg(customer_total_orders=("Order Id","count"),customer_late_rate=("Late_delivery_risk","mean"),customer_avg_delay_gap=("delay_gap","mean")).reset_index()

# Merge back into main dataframe
df=df.merge(customer_stats,on="Order Customer Id",how="left")

print("New Columns Added :")
customer_stats.describe()

New Columns Added :


,Order Customer Id,customer_total_orders,customer_late_rate,customer_avg_delay_gap
count,20649.000000,20649.000000,20649.000000,20649.000000
mean,10399.579205,8.742118,0.548731,0.568694
std,5993.232139,8.375983,0.383953,1.152474
min,1.000000,1.000000,0.000000,-2.000000
25%,5208.000000,1.000000,0.185185,0.000000
50%,10406.000000,7.000000,0.578947,0.666667
75%,15594.000000,15.000000,1.000000,1.034483
max,20757.000000,47.000000,1.000000,4.000000


## Feature: Customer-Level Aggregations

For each customer we compute:
- customer_total_orders: how active they are
- customer_late_rate: what % of their past orders were late
- customer_avg_delay_gap: their average delay in days

A customer with a high historical late rate likely places orders through
high-risk shipping modes or problematic regions.

In [52]:
df['Price Tier']=pd.qcut(df['Product Price'],q=4,labels=[0,1,2,3]).astype(int)
print(df['Price Tier'].value_counts())

Price Tier
0    64441
2    53476
3    36687
1    25912
Name: count, dtype: int64


## Feature: price_tier

Product price binned into 4 quartile-based tiers (0=cheapest, 3=most expensive).
Higher value items may receive different handling or carrier priority.

In [53]:
from sklearn.preprocessing import LabelEncoder

In [54]:
cat_cols = ['Market', 'Order Region', 'Customer Segment', 
            'Department Name', 'Shipping Mode', 'Type','Order Status']
le=LabelEncoder()
for cat in cat_cols:
    df[cat+"_encoded"]=le.fit_transform(df[cat].astype(str))

print("Encoded Columns Added ")

Encoded Columns Added 


## Categorical Encoding

Label encoding applied to categorical columns. We keep the originals
for reference and add _encoded versions for the model.
Note: For tree-based models like XGBoost, label encoding is acceptable
as the model handles non-ordinal categoricals well internally.

In [55]:
cols_to_drop = [
    'order date (DateOrders)', 'shipping date (DateOrders)',
    'Order Id', 'Order Customer Id', 'Order Item Id',
    'Order Item Cardprod Id', 'Product Card Id', 'Product Category Id',
    'Category Id', 'Department Id', 'Customer Id',
    'Customer City', 'Customer Country', 'Customer State',
    'Order City', 'Order Country', 'Order State',
    'Product Name', 'Category Name',
    'Market', 'Order Region', 'Customer Segment',
    'Department Name', 'Shipping Mode', 'Type',
    'order_month', 'Order_day_of_week',"Order Status"
]

df.drop(columns=cols_to_drop, inplace=True)
print(f"Final shape: {df.shape}")
print(df.columns.tolist())

Final shape: (180516, 33)
['Days for shipping (real)', 'Days for shipment (scheduled)', 'Benefit per order', 'Sales per customer', 'Late_delivery_risk', 'Latitude', 'Longitude', 'Order Item Discount', 'Order Item Discount Rate', 'Order Item Product Price', 'Order Item Profit Ratio', 'Order Item Quantity', 'Sales', 'Order Item Total', 'Order Profit Per Order', 'Product Price', 'Product Status', 'delay_gap', 'Order_processing_days', 'is_holiday_window', 'is_weekend_order', 'Shipping_mode_risk', 'customer_total_orders', 'customer_late_rate', 'customer_avg_delay_gap', 'Price Tier', 'Market_encoded', 'Order Region_encoded', 'Customer Segment_encoded', 'Department Name_encoded', 'Shipping Mode_encoded', 'Type_encoded', 'Order Status_encoded']


## Columns Dropped for Modelling

Removed: ID columns (no predictive value), raw date columns (already
extracted features from them), original categorical columns (replaced
by encoded versions), and geography strings (replaced by encoded versions).

What remains is a clean numeric feature matrix ready for train/test split.

In [56]:
df.to_csv('dataset/processed/feature_data.csv', index=False)